In [11]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib widget

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [12]:
exports_folder = Path("refine_plus_export_pool")

In [13]:
from pathlib import Path
from typing import List, Dict

import igl
import numpy as np
import polars as pl
from polars import Series
from polars.dataframe.frame import DataFrame

from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import VNIR_MEASUREMENT_NAMES, VNIR_SIGMA_MEASUREMENT_NAMES
from boulder_statistics.refinement_plus.facet_parser import FacetParser
from boulder_statistics.refinement_plus.bulk_parse_data_vnir_maps import (
    FACET_SHAPE_MODELS,
    DataVnirMaps,
)
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools


mesh_folder = Path(
    r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models"
)

measurement_types: list[str] = VNIR_MEASUREMENT_NAMES + VNIR_SIGMA_MEASUREMENT_NAMES

data_vnir_maps: DataFrame = DataVnirMaps.bulk_parse(ed)
mission_phases: Series = data_vnir_maps["mission_phase"].unique()


for mission_phase in mission_phases:
    vnir_spectral_export_path: Path = exports_folder / f"02_add_vnir_maps_{mission_phase}"
    if vnir_spectral_export_path.exists():
        continue

    print(f"Running for mission phase {mission_phase}")
    mesh_file_path: Path = mesh_folder / FACET_SHAPE_MODELS[mission_phase]

    # Load original mesh arrays for libigl
    V, F = igl.read_triangle_mesh(mesh_file_path)

    # Load processed mesh tables
    points, tris = FacetParser.load_mesh(mesh_file_path)

    facets: DataFrame = data_vnir_maps.filter(
        pl.col("mission_phase") == mission_phase
    )

    # ------------------------------------------------------------
    # Associate facets with mesh triangles using point-to-mesh distance
    # ------------------------------------------------------------

    P = (
        facets
        .select(["x", "y", "z"])
        .to_numpy()
        .astype(np.float64)
    )

    sqrD, tri_idx, closest_points = igl.point_mesh_squared_distance(
        P,
        V,
        F
    )

    associate_distance = np.sqrt(sqrD)

    facet_nums = (
        facets
        .get_column("facet_num")
        .to_numpy()
        .astype(np.int32)
    )

    facet_nums_to_tri_nums_lookup = pl.DataFrame(
        {
            "facet_num": facet_nums,
            "tri_num": tri_idx.astype(np.int32),
            "associate_distance": associate_distance,
        }
    )

    # Sanity checks
    if not np.all(
        facet_nums_to_tri_nums_lookup["associate_distance"].to_numpy()
        < 1e-5
    ):
        print("Facets not lining up, skipping")
        continue

    duplicates = (
        facet_nums_to_tri_nums_lookup
        .group_by("tri_num")
        .len()
        .filter(pl.col("len") > 1)
    )

    if duplicates.height != 0:
        print("duplicate triangles found!, skipping")
        print(duplicates)

        continue


    # ------------------------------------------------------------
    # Join facet measurements onto triangles
    # ------------------------------------------------------------

    tris_facets_joined: DataFrame = (
        tris
        .join(
            facet_nums_to_tri_nums_lookup,
            on="tri_num"
        )
        .join(
            facets,
            on="facet_num"
        )
    )

    if tris_facets_joined["tri_num"].n_unique() != tris_facets_joined.height:
        print("Non unique tris detected, skipping")
        continue


    # ------------------------------------------------------------
    # Create triangle-indexed measurement arrays
    # ------------------------------------------------------------

    measurement_type_triangle_values_dict: Dict[str, np.ndarray] = {}

    for measurement_type in measurement_types:

        triangle_values = np.full(
            tris.height,
            np.nan,
            dtype=np.float32,
        )

        triangle_values[
            tris_facets_joined["tri_num"].to_numpy()
        ] = (
            tris_facets_joined
            .get_column(measurement_type)
            .to_numpy()
            .astype(np.float32)
        )

        measurement_type_triangle_values_dict[measurement_type] = (
            triangle_values
        )


    # ------------------------------------------------------------
    # Rasterisation
    # ------------------------------------------------------------

    def handle_chunk(chunk: QCubeChunk) -> List[np.ndarray]:

        triangle_index_image: np.ndarray = FacetParser.rasterize_facets(
            points,
            tris,
            chunk,
        )

        measurement_types_results: List[np.ndarray] = []

        mask = ~np.isnan(triangle_index_image)

        for measurement_type in measurement_types:

            output = np.full(
                triangle_index_image.shape,
                np.nan,
                dtype=np.float32,
            )

            output[mask] = (
                measurement_type_triangle_values_dict[measurement_type]
                [
                    triangle_index_image[mask]
                    .astype(np.int32)
                ]
            )

            measurement_types_results.append(output.T)

        return measurement_types_results


    ChunkingTools.bulk_append_by_chunks(
        dp.combined_atlas.select("i", "j", "face"),
        vnir_spectral_export_path,
        [f"{mission_phase} {type_name}" for type_name in measurement_types],
        handle_chunk,
        chunks=QCubeChunk.generate(depth=2),
    )

Running for mission phase reconb
Facets not lining up, skipping
